# Parkinson's Tremor Screening Model Training & Analysis
This notebook covers the interactive development, data loading, augmentation, and training of Machine Learning models used for screening Parkinson's Tremors.

## 1. Imports and Configurations

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Load Clinical Dataset
We load the clean clinical patient records from the `dataset` folder in the project root.

In [ ]:
dataset_path = os.path.join("..", "dataset", "Parkinsons_Tremor_Clean_Dataset.csv")
df = pd.read_csv(dataset_path)
print(f"Loaded {len(df)} patient records.")
df.head()

## 3. Data Augmentation
To build a robust classifier, we augment the clinical dataset. Different clinical profiles have varying ranges of tremor frequency (Hz) and scaled amplitude, as captured during patient sessions.

In [ ]:
np.random.seed(42)

freqs = []
amps = []
y_severity = []
y_category = []
y_stage = []

diag_aug_map = {
    "Parkinson's": 40,
    "Healthy": 140,
    "Other Movement Disorders": 180,
    "Essential Tremor": 390,
    "Atypical Parkinsonism": 730,
    "Multiple Sclerosis": 1000
}
for _, row in df.iterrows():
    diagnosis = str(row.get('diagnosis', 'Healthy')).strip()
    tremor_type = str(row.get('tremor_text_type', 'not_mentioned')).strip()
    
    n_augs = diag_aug_map.get(diagnosis, 40)
    for _ in range(n_augs):
        if diagnosis == 'Healthy':
            f = np.random.uniform(0.0, 3.0)
            a = np.random.uniform(0.1, 1.4)
        elif diagnosis == "Parkinson's":
            if tremor_type in ['resting', 'tremor_dominant']:
                f = np.random.uniform(4.0, 6.0)
                a = np.random.uniform(2.0, 25.0)
            else:
                f = np.random.uniform(4.0, 7.0)
                a = np.random.uniform(1.5, 10.0)
        elif 'Essential' in tremor_type or diagnosis == 'Other Movement Disorders':
            f = np.random.uniform(8.0, 12.0)
            a = np.random.uniform(1.5, 15.0)
        else:
            f = np.random.uniform(1.0, 15.0)
            a = np.random.uniform(0.5, 4.0)
            
        freqs.append(f)
        amps.append(a)

        if f < 3.0 or a < 1.5:
            sev = "Normal"
        elif 1.5 <= a < 5.0:
            sev = "Mild"
        elif 5.0 <= a <= 15.0:
            sev = "Moderate"
        else:
            sev = "Severe"
        y_severity.append(sev)

        if 4.0 <= f <= 6.0 and diagnosis == "Parkinson's":
            cat = "Parkinsonian Rest Tremor Range (4-6 Hz)"
        elif 8.0 <= f <= 12.0:
            cat = "Essential / Physiological Tremor Range (8-12 Hz)"
        elif diagnosis == "Healthy":
            cat = "Normal / Low Activity"
        else:
            cat = "Mixed / Unspecified Tremor"
        y_category.append(cat)

        stg = "Stage 0 (No Tremor)"
        if diagnosis == "Parkinson's":
            if sev == "Mild":
                stg = "Stage 1 (Unilateral involvement only)"
            elif sev == "Moderate":
                stg = "Stage 2 (Bilateral involvement, without impairment of balance)"
            elif sev == "Severe":
                stg = "Stage 3 (Mild to moderate bilateral disease; some postural instability)"
        y_stage.append(stg)

X = np.stack([freqs, amps], axis=1)
print(f"Dataset augmented to {X.shape[0]} samples.")

## 4. Train-Test Splits

In [ ]:
y = np.column_stack([y_severity, y_category, y_stage])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 5. Model Training & Evaluation

In [ ]:
tremor_clf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
tremor_clf.fit(X_train, y_train)
preds = tremor_clf.predict(X_test)
sev_preds = preds[:, 0]
cat_preds = preds[:, 1]
stg_preds = preds[:, 2]
print(f"Severity Classifier Accuracy: {accuracy_score(y_test[:, 0], sev_preds) * 100:.2f}%")
print(classification_report(y_test[:, 0], sev_preds))

In [ ]:
print(f"Category Classifier Accuracy: {accuracy_score(y_test[:, 1], cat_preds) * 100:.2f}%")
print(classification_report(y_test[:, 1], cat_preds))

In [ ]:
print(f"Stage Classifier Accuracy: {accuracy_score(y_test[:, 2], stg_preds) * 100:.2f}%")
print(classification_report(y_test[:, 2], stg_preds))

## 6. Plotting Augmented Patient Distribution

In [ ]:
plt.figure(figsize=(10, 6))
scatter_df = pd.DataFrame({
    'Frequency': X[:, 0],
    'Amplitude': X[:, 1],
    'Severity': y_severity
})

for name, group in scatter_df.groupby('Severity'):
    plt.scatter(group['Frequency'], group['Amplitude'], label=name, alpha=0.6, edgecolors='k')

plt.title("Augmented Patient Clinical Distribution (Frequency vs. Amplitude)")
plt.xlabel("Dominant Frequency (Hz)")
plt.ylabel("Tremor Amplitude")
plt.legend(title="Severity Level")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()